In [5]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
 
SEED       = 42
TARGET     = 'demand_label'
MODEL_TYPE = 'boost'
DATA_DIR   = '../../data/splits/'
# DATA_DIR = '../data/splits/'
 
train_df = pd.read_csv(DATA_DIR + 'train.csv', low_memory=False)
test_df  = pd.read_csv(DATA_DIR + 'test.csv',  low_memory=False)
 
DROP_COLS  = [
    'demand_label', 'demand_label_3', 'demand_score',
    'Price_log', 'Price_original',
]
DROP_COLS += [c for c in train_df.columns if c.endswith('_raw')]
DROP_COLS += [c for c in train_df.columns if 'Parsed_Amenities' in c]
 
feature_cols = [c for c in train_df.columns if c not in DROP_COLS]
 
# Sanitize column names (required for LightGBM)
def sanitize(cols):
    return {c: re.sub(r'[^A-Za-z0-9_]+', '_', c) for c in cols}
 
rename_map   = sanitize(feature_cols)
train_df     = train_df.rename(columns=rename_map)
test_df      = test_df.rename(columns=rename_map)
feature_cols = list(rename_map.values())
 
X_all  = train_df[feature_cols]
y_all  = train_df[TARGET]
X_test = test_df[feature_cols]
y_test = test_df[TARGET]
 
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
)
 
# Class imbalance weight (used in XGBoost and LightGBM)
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos if pos > 0 else 1.0
 
print(f'Train : {X_train.shape} | Val : {X_val.shape} | Test : {X_test.shape}')
print(f'Class balance — train : {y_train.value_counts(normalize=True).round(3).to_dict()}')
print(f'Class balance — val   : {y_val.value_counts(normalize=True).round(3).to_dict()}')
print(f'Class balance — test  : {y_test.value_counts(normalize=True).round(3).to_dict()}')
print(f'Remaining nulls — train: {X_train.isna().sum().sum()}')
print(f'Remaining nulls — test : {X_test.isna().sum().sum()}')
print(f'Features used : {len(feature_cols)}')
print(f'scale_pos_weight = {scale_pos_weight:.4f}')

Train : (311661, 95) | Val : (77916, 95) | Test : (97395, 95)
Class balance — train : {0: 0.5, 1: 0.5}
Class balance — val   : {0: 0.5, 1: 0.5}
Class balance — test  : {0: 0.5, 1: 0.5}
Remaining nulls — train: 0
Remaining nulls — test : 0
Features used : 95
scale_pos_weight = 1.0007


In [6]:
import xgboost as xgb

model = xgb.XGBClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Top 20 features by importance
importance = pd.Series(model.feature_importances_, index=feature_cols)
print(importance.sort_values(ascending=False).head(20))

Days_Since_Last_Review                   0.174683
Host_Response_Time_Unknown               0.123570
Amenity_Shampoo                          0.054123
Host_Response_Time_within_an_hour        0.053412
Amenity_Air_conditioning                 0.049046
Amenity_Family_kid_friendly              0.036662
Amenity_Smoke_detector                   0.030048
Review_Scores_Rating                     0.029044
Minimum_Nights                           0.028706
Jurisdiction_Names                       0.026907
Amenity_Hangers                          0.024993
Host_Response_Time_within_a_few_hours    0.022001
Review_Scores_Value                      0.018903
Review_Scores_Composite                  0.016761
Amenity_Essentials                       0.016158
Longitude                                0.013714
Review_Scores_Cleanliness                0.013160
Review_Scores_Location                   0.012606
Days_Since_First_Review                  0.011524
Amenity_Laptop_friendly_workspace        0.011290


In [7]:
# Check correlation of every feature with the target
correlations = X_train.corrwith(y_train).abs().sort_values(ascending=False)
print(correlations.head(20))

# Any feature with correlation > 0.95 with the label is essentially the label
print("\nFeatures with |correlation| > 0.95 with demand_label:")
print(correlations[correlations > 0.95])

Host_Response_Time_Unknown           0.427904
Host_Response_Time_within_an_hour    0.353005
Days_Since_Last_Review               0.305142
Amenities_Count                      0.241766
Cancellation_Policy_flexible         0.219037
Amenity_Shampoo                      0.214561
Amenity_Hangers                      0.196619
Host_Verifications_Count             0.195072
Amenity_Hair_dryer                   0.190050
Amenity_Iron                         0.173204
Host_Identity_Verified               0.151194
Amenity_Laptop_friendly_workspace    0.149661
Amenity_Essentials                   0.144805
Amenity_Carbon_monoxide_detector     0.140119
Minimum_Nights                       0.134466
Review_Scores_Value                  0.129320
Review_Scores_Location               0.119321
Amenity_Heating                      0.117706
Cancellation_Policy_moderate         0.115054
Guests_Included                      0.110165
dtype: float64

Features with |correlation| > 0.95 with demand_label:
Series([],

In [8]:
# Check if any leaky column survived into your feature set
leaky_suspects = [
    'Number_of_Reviews', 'Reviews_per_Month',
    'Availability_365', 'Availability_30',
    'Availability_60', 'Availability_90',
    'demand_score', 'Number_of_Reviews_raw',
    'Reviews_per_Month_raw', 'Availability_365_raw',
]

found = [c for c in leaky_suspects if c in feature_cols]
print(f"Leaky columns still in features: {found}")

# Also check for any column with 'review' or 'availab' in the name
suspicious = [c for c in feature_cols 
              if any(k in c.lower() for k in ['review', 'availab', 'demand'])]
print(f"\nSuspicious column names in features:\n{suspicious}")

Leaky columns still in features: []

Suspicious column names in features:
['Review_Scores_Rating', 'Review_Scores_Accuracy', 'Review_Scores_Cleanliness', 'Review_Scores_Checkin', 'Review_Scores_Communication', 'Review_Scores_Location', 'Review_Scores_Value', 'Days_Since_First_Review', 'Days_Since_Last_Review', 'Review_Scores_Composite']


In [9]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.dummy import DummyClassifier

# 1. Dummy baseline — what does random chance get?
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
print(f"Dummy accuracy : {dummy.score(X_test, y_test):.4f}")

# 2. Your actual model on test
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]
print(f"Model accuracy : {model.score(X_test, y_test):.4f}")
print(f"ROC-AUC        : {roc_auc_score(y_test, y_pred_prob):.4f}")
print(classification_report(y_test, y_pred))

# 3. Compare train vs test accuracy — large gap = overfitting, not leakage
print(f"Train accuracy : {model.score(X_train, y_train):.4f}")
print(f"Test accuracy  : {model.score(X_test, y_test):.4f}")
print(f"Gap            : {model.score(X_train, y_train) - model.score(X_test, y_test):.4f}")

Dummy accuracy : 0.5002
Model accuracy : 0.9149
ROC-AUC        : 0.9765
              precision    recall  f1-score   support

           0       0.93      0.90      0.91     48714
           1       0.90      0.93      0.92     48681

    accuracy                           0.91     97395
   macro avg       0.92      0.91      0.91     97395
weighted avg       0.92      0.91      0.91     97395

Train accuracy : 0.9286
Test accuracy  : 0.9149
Gap            : 0.0137
